In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from notebookutils import *
from pyspark.sql import functions as F
from delta.tables import *
from notebookutils import mssparkutils

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, 3, Finished, Available, Finished, False)

In [2]:
tables_str = tables_str
bronze = bronze
lakehouse = lakehouse
workspace = workspace
silver = silver

Tables = [t.strip() for t in tables_str.split(",")]
Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, 4, Finished, Available, Finished, False)

NameError: name 'tables_str' is not defined

In [ ]:
table = None
for i in Tables:
    if i.lower() == "encounter":
        table = i
        break

if table is None:
    mssparkutils.notebook.exit("Exiting notebook: Table is not encounter")

print(table + " table is present ")

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
bronze_path = f"Files/{bronze}/{table.lower()}"
silver_path = f"Files/{silver}/{table.lower()}"

# Check if silver path exists
silver_exists = mssparkutils.fs.exists(silver_path)

# Read bronze data
df = spark.read.format("delta").load(bronze_path)

# Apply filter only if silver exists
if silver_exists:
    df = df.filter(
        col("load_date") >= date_sub(current_date(), 4)
    )

df.printSchema()

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
final_df = (
    df.select("api_url", "extraction_timestamp", "load_date", "raw_json.entry")

    # Step 1: clean outer brackets
    .withColumn("entry_clean", F.regexp_replace("entry", r"^\[|\]$", ""))

    # Step 2: split records
    .withColumn("entry_array", F.split("entry_clean", r"\},\s*\{"))

    # Step 3: explode
    .withColumn("entry_item", F.explode("entry_array"))

    # Step 4: normalize record
    .withColumn("entry_item", F.concat(F.lit("{"), F.col("entry_item"), F.lit("}")))

    # Step 5: extract ALL important fields
    .select(

        # ===== Base metadata =====
        "api_url",
        "extraction_timestamp",
        "load_date",

        # ===== Encounter identifiers =====
        F.regexp_extract("entry_item", r"id=([^,}]+)", 1).alias("encounter_id"),
        F.regexp_extract("entry_item", r"resourceType=([^,}]+)", 1).alias("resource_type"),
        F.regexp_extract("entry_item", r"status=([^,}]+)", 1).alias("status"),
        F.regexp_extract("entry_item", r"fullUrl=([^,}]+)", 1).alias("full_url"),

        # ===== Class =====
        F.regexp_extract("entry_item", r"class=\{code=([^,}]+)", 1).alias("class_code"),
        F.regexp_extract("entry_item", r"class=\{code=[^,}]+,.*display=([^,}]+)", 1).alias("class_display"),

        # ===== Patient =====
        F.regexp_extract("entry_item", r"subject=\{reference=([^,}]+)", 1).alias("patient_reference"),
        F.regexp_extract("entry_item", r"subject=\{reference=[^,}]+, display=([^,}]+)", 1).alias("patient_display"),

        # ===== Encounter period =====
        F.regexp_extract("entry_item", r"start=([^,}]+)", 1).alias("start_date"),
        F.regexp_extract("entry_item", r"end=([^,}]+)", 1).alias("end_date"),

        # ===== Reason (text + SNOMED if present) =====
        F.regexp_extract("entry_item", r"text=([^,}]+)", 1).alias("reason_text"),
        F.regexp_extract("entry_item", r"code=([0-9]+)", 1).alias("reason_code"),
        F.regexp_extract("entry_item", r"display=([^,}]+)", 1).alias("reason_display"),

        # ===== Priority =====
        F.regexp_extract("entry_item", r"priority=\{coding=\[\{code=([^,}]+)", 1).alias("priority_code"),
        F.regexp_extract("entry_item", r"priority=.*display=([^,\}]+)", 1).alias("priority_display"),

        # ===== Type =====
        F.regexp_extract("entry_item", r"type=\[\{coding=\[\{code=([^,}]+)", 1).alias("type_code"),
        F.regexp_extract("entry_item", r"type=.*display=([^,\}]+)", 1).alias("type_display"),

        # ===== Identifier =====
        F.regexp_extract("entry_item", r"identifier=\[\{system=([^,}]+)", 1).alias("identifier_system"),
        F.regexp_extract("entry_item", r"value=([^,}]+)", 1).alias("identifier_value"),

        # ===== Length of stay =====
        F.regexp_extract("entry_item", r"length=\{unit=([^,}]+)", 1).alias("length_unit"),
        F.regexp_extract("entry_item", r"length=.*value=([0-9]+)", 1).alias("length_value"),

        # ===== Hospitalization =====
        F.regexp_extract("entry_item", r"preAdmissionIdentifier=\{system=([^,}]+)", 1).alias("pre_admission_system"),
        F.regexp_extract("entry_item", r"preAdmissionIdentifier=.*value=([^,}]+)", 1).alias("pre_admission_value"),

        F.regexp_extract("entry_item", r"admitSource=.*code=([^,}]+)", 1).alias("admit_source_code"),
        F.regexp_extract("entry_item", r"admitSource=.*display=([^,}]+)", 1).alias("admit_source_display"),

        F.regexp_extract("entry_item", r"dischargeDisposition=.*code=([^,}]+)", 1).alias("discharge_code"),
        F.regexp_extract("entry_item", r"dischargeDisposition=.*display=([^,}]+)", 1).alias("discharge_display"),

        # ===== Meta =====
        F.regexp_extract("entry_item", r"lastUpdated=([^,}]+)", 1).alias("last_updated"),
        F.regexp_extract("entry_item", r"versionId=([^,}]+)", 1).alias("version_id"),
        F.regexp_extract("entry_item", r"source=#([^,}]+)", 1).alias("source_system"),

        # ===== Participant =====
        F.regexp_extract("entry_item", r"participants.*Individual\}\}\]\}\},\s*reference=([^,}]+)", 1).alias("participant_reference")

    )
)

display(final_df.limit(5))
silver_condition = final_df

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
# -----------------------------
#  Create IDs FIRST (before drop)
# -----------------------------
silver_condition_fixed = (
    silver_condition
    .withColumn(
        "patient_id",
        trim(split(col("patient_reference"), "/")[1])
    )
)

# -----------------------------
# Drop old columns safely
# -----------------------------
silver_condition_fixed = silver_condition_fixed.drop(
    "patient_reference"
)

# -----------------------------
#  Fix timestamp type (important for ordering)
# -----------------------------
silver_condition_fixed = silver_condition_fixed.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
#  Display sorted output
# -----------------------------
display(
    silver_condition_fixed.orderBy(col("last_updated")).limit(100)
)

silver_condition = silver_condition_fixed

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:

# -----------------------------
# Step 1: Ensure timestamp type
# -----------------------------
silver_encounter = silver_condition.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
# Step 2: Define window (latest record per encounter + patient)
# -----------------------------
window_spec = Window.partitionBy(
    "encounter_id"
).orderBy(
    col("last_updated").desc()
)

# -----------------------------
# Step 3: Deduplicate using row_number
# -----------------------------
silver_encounter = (
    silver_encounter
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# -----------------------------
# Step 4: Display result (Fabric-safe)
# -----------------------------
display(silver_encounter.limit(5))

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
staging_df = silver_encounter \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_path = f"Files/{silver}/{table.lower()}"

if not mssparkutils.fs.exists(silver_path):
    staging_df.write.format("delta").mode("overwrite").save(silver_path)

    mssparkutils.notebook.exit("Exiting notebook: First time load completed")
else:
    print("Table exists")

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
staging_df.createOrReplaceTempView("staging_encounter")

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
query = f"""
MERGE INTO delta.`{silver_path}` AS tgt
USING staging_encounter AS src
ON tgt.encounter_id = src.encounter_id
AND tgt.is_current = true

-- 1. EXPIRE OLD RECORD WHEN DATA CHANGES
WHEN MATCHED AND (
    COALESCE(tgt.status, '') <> COALESCE(src.status, '') OR
    COALESCE(tgt.class_code, '') <> COALESCE(src.class_code, '') OR
    COALESCE(tgt.class_display, '') <> COALESCE(src.class_display, '') OR
    COALESCE(tgt.patient_id, '') <> COALESCE(src.patient_id, '') OR
    COALESCE(tgt.start_date, '') <> COALESCE(src.start_date, '') OR
    COALESCE(tgt.end_date, '') <> COALESCE(src.end_date, '') OR
    COALESCE(tgt.priority_code, '') <> COALESCE(src.priority_code, '') OR
    COALESCE(tgt.type_code, '') <> COALESCE(src.type_code, '') OR
    COALESCE(tgt.reason_code, '') <> COALESCE(src.reason_code, '') OR
    COALESCE(tgt.admit_source_code, '') <> COALESCE(src.admit_source_code, '') OR
    COALESCE(tgt.discharge_code, '') <> COALESCE(src.discharge_code, '')
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

-- 2. INSERT NEW VERSION WHEN NO ACTIVE MATCH EXISTS
WHEN NOT MATCHED
THEN INSERT (
    api_url,
    extraction_timestamp,
    load_date,
    encounter_id,
    resource_type,
    status,
    full_url,
    class_code,
    class_display,
    patient_display,
    start_date,
    end_date,
    reason_text,
    reason_code,
    reason_display,
    priority_code,
    priority_display,
    type_code,
    type_display,
    identifier_system,
    identifier_value,
    length_unit,
    length_value,
    pre_admission_system,
    pre_admission_value,
    admit_source_code,
    admit_source_display,
    discharge_code,
    discharge_display,
    last_updated,
    version_id,
    source_system,
    participant_reference,
    patient_id,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.api_url,
    src.extraction_timestamp,
    src.load_date,
    src.encounter_id,
    src.resource_type,
    src.status,
    src.full_url,
    src.class_code,
    src.class_display,
    src.patient_display,
    src.start_date,
    src.end_date,
    src.reason_text,
    src.reason_code,
    src.reason_display,
    src.priority_code,
    src.priority_display,
    src.type_code,
    src.type_display,
    src.identifier_system,
    src.identifier_value,
    src.length_unit,
    src.length_value,
    src.pre_admission_system,
    src.pre_admission_value,
    src.admit_source_code,
    src.admit_source_display,
    src.discharge_code,
    src.discharge_display,
    src.last_updated,
    src.version_id,
    src.source_system,
    src.participant_reference,
    src.patient_id,
    current_timestamp(),
    NULL,
    true
)
"""

query1 = f"""
INSERT INTO delta.`{silver_path}` (
    api_url,
    extraction_timestamp,
    load_date,
    encounter_id,
    resource_type,
    status,
    full_url,
    class_code,
    class_display,
    patient_display,
    start_date,
    end_date,
    reason_text,
    reason_code,
    reason_display,
    priority_code,
    priority_display,
    type_code,
    type_display,
    identifier_system,
    identifier_value,
    length_unit,
    length_value,
    pre_admission_system,
    pre_admission_value,
    admit_source_code,
    admit_source_display,
    discharge_code,
    discharge_display,
    last_updated,
    version_id,
    source_system,
    participant_reference,
    patient_id,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.api_url,
    src.extraction_timestamp,
    src.load_date,
    src.encounter_id,
    src.resource_type,
    src.status,
    src.full_url,
    src.class_code,
    src.class_display,
    src.patient_display,
    src.start_date,
    src.end_date,
    src.reason_text,
    src.reason_code,
    src.reason_display,
    src.priority_code,
    src.priority_display,
    src.type_code,
    src.type_display,
    src.identifier_system,
    src.identifier_value,
    src.length_unit,
    src.length_value,
    src.pre_admission_system,
    src.pre_admission_value,
    src.admit_source_code,
    src.admit_source_display,
    src.discharge_code,
    src.discharge_display,
    src.last_updated,
    src.version_id,
    src.source_system,
    src.participant_reference,
    src.patient_id,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_encounter src
JOIN delta.`{silver_path}` tgt
    ON tgt.encounter_id = src.encounter_id
WHERE tgt.is_current = false
AND (
    COALESCE(tgt.status, '') <> COALESCE(src.status, '') OR
    COALESCE(tgt.class_code, '') <> COALESCE(src.class_code, '') OR
    COALESCE(tgt.patient_id, '') <> COALESCE(src.patient_id, '') OR
    COALESCE(tgt.start_date, '') <> COALESCE(src.start_date, '') OR
    COALESCE(tgt.end_date, '') <> COALESCE(src.end_date, '')
)
AND NOT EXISTS (
    SELECT 1
    FROM delta.`{silver_path}` t2
    WHERE t2.encounter_id = src.encounter_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)

In [ ]:
df = spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`") \
    .orderBy("version", ascending=False) \
    .limit(2)

display(df)

StatementMeta(, 0a5ba449-7137-43ba-8518-adcd146044c6, -1, Cancelled, , Cancelled, True)